In [3]:
pip install tavily-python


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from tavily import TavilyClient
tavily = TavilyClient(api_key="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")

result = tavily.search("today's weather in Dublin")
print(result)


{'query': "today's weather in Dublin", 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in Dublin', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Dublin', 'region': 'Dublin', 'country': 'Ireland', 'lat': 53.3331, 'lon': -6.2489, 'tz_id': 'Europe/Dublin', 'localtime_epoch': 1781275083, 'localtime': '2026-06-12 15:38'}, 'current': {'last_updated_epoch': 1781274600, 'last_updated': '2026-06-12 15:30', 'temp_c': 17.4, 'temp_f': 63.3, 'is_day': 1, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/day/116.png', 'code': 1003}, 'wind_mph': 19.9, 'wind_kph': 32.0, 'wind_degree': 268, 'wind_dir': 'W', 'pressure_mb': 1018.0, 'pressure_in': 30.06, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 55, 'cloud': 75, 'feelslike_c': 17.4, 'feelslike_f': 63.3, 'windchill_c': 17.2, 'windchill_f': 62.9, 'heatindex_c': 17.2, 'heatindex_f': 62.9, 'dewpoint_c': 9.5, 'dewpoint_f': 49.0, 'vis_km': 10.0, 'vis_mile

In [6]:
# define a def to search the web
def search_web(query: str) -> str:
    # use Tavily to search the web
    results = tavily.search(query, max_results=3)

    output = []
    for result in results:
        output.append(f"title: {result['title']} \n content: { result['content']} \n url: {result['url']}\n")
    return output


In [8]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage
from typing import Annotated
from typing_extensions import TypedDict
import operator

# 1. Define the StateGraph
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

# 2. initialize the Model
model = ChatOpenAI(api_key="api_key",model="gpt-4")

# 3. the new tool
def search_web(query: str) -> str:
    results = tavily.search(query, max_results=3)
    output = []

    for result in results:
        output.append(f"title : {result['title']} \n content: {result['content']}\n")

    return "\n".join(output)

# 4. the tool intruction
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "search info from web, get the newest information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "search the key word"
                    }
                },
                "required": ["query"]
            }
        }
    }
]
tool_map = {"search_web": search_web}
model_with_tools = model.bind_tools(tools_schema)

# 5. define the node
def llm_node(state: AgentState):
    response = model_with_tools.invoke(state["messages"])
    return {message: [response]}

def tool_node(state: AgentState):
    last_message = state["messages"][-1]
    tool_results = []

    for tool_call in last_message.tool_calls:
        name = tool_call["name"]
        args = tool_Call["args"]

        print(f"the key word is {args}")

        result = tool_map[name](**args)
        tool_results.append(
            ToolMessage(
                ToolMessage(content=result, tool_call_id=tool_call["id"])
            )
        )
    return {message: tool_results}

def should_continue(state: AgentState):
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "call_tool"
    return "end"

# 6. define the graph
graph_builder = StateGraph(AgentState)
graph_builder.add_node("llm", llm_node)
graph_builder.add_node("tool", tool_node)
graph_builder.set_entry_point("llm")
graph_builder.add_conditional_edges(
    "llm", should_continue,
    {"call_tool": "tool", "end": END}
)
graph_builder.add_edge("tool", "llm")
agent = graph_builder.compile()

# 7. run the agent
res = agent.invoke({"messages": [HumanMessage(content="what is the weather in New York?")]})

print(f" the final result is: {res["messages"][-1].content}")

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: api_key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}